# Under the Odaa — Wan 2.2 TI2V-5B Image-to-Video (Kaggle T4)
Settings: **Accelerator: GPU T4 x2**, **Internet: On**.
Saves each 480p clip to `outputs/wan/shot_XX.mp4` as it finishes (sessions time out) and logs latency/GPU minutes.

In [ ]:
REPO_URL = "https://github.com/christianwondeson/odaa.git"
BRANCH = "improve/ethiopian-characters-and-arc"
!git clone -b {BRANCH} {REPO_URL} odaa || (cd odaa && git pull)
%cd odaa
!pip -q install -U diffusers transformers accelerate imageio-ffmpeg hf_transfer

In [ ]:
import json, os, pathlib, time, torch
from PIL import Image
from diffusers import AutoencoderKLWan, WanImageToVideoPipeline
from diffusers.utils import export_to_video

sb = json.loads(pathlib.Path("storyboard.json").read_text(encoding="utf-8"))
out_dir = pathlib.Path("outputs/wan"); out_dir.mkdir(parents=True, exist_ok=True)

model_id = "Wan-AI/Wan2.2-TI2V-5B-Diffusers"
vae = AutoencoderKLWan.from_pretrained(model_id, subfolder="vae", torch_dtype=torch.bfloat16)
pipe = WanImageToVideoPipeline.from_pretrained(model_id, vae=vae, torch_dtype=torch.bfloat16)
pipe.enable_model_cpu_offload()  # Required to fit on 16 GB T4 VRAM
pipe.vae.enable_tiling()

for s in sb["shots"]:
    sid = s["id"]
    dst = out_dir / f"shot_{sid}.mp4"
    if dst.exists():
        print(f"shot {sid}: cached"); continue
    img = Image.open(f"outputs/keyframes/shot_{sid}.jpg").convert("RGB").resize((832, 480))
    t0 = time.time()
    gen = torch.Generator("cpu").manual_seed(int(s["seed"]))
    frames = pipe(
        image=img,
        prompt=s["motion_prompt"],
        negative_prompt="blurry, distorted face, extra limbs, flicker, text, watermark",
        height=480, width=832, num_frames=49, guidance_scale=5.0, generator=gen
    ).frames[0]
    export_to_video(frames, str(dst), fps=16)
    dt = round(time.time() - t0, 2)
    print(f"shot {sid}: saved {dst} ({dt}s / {dt/60:.2f} GPU min)")

In [ ]:
!zip -r /kaggle/working/wan_clips.zip outputs/wan/
print("Download /kaggle/working/wan_clips.zip and extract into outputs/wan/ locally.")